Support vector machines, the kernel trick, and regularization for support vector machines

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm 
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

c:\Users\caleb\anaconda3\envs\iris-fisher\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
random_state=42

In [3]:
classification_reports = []
classification_report_keys = []

## Data

In [4]:
# Import IBM dataset
df = pd.read_csv('C:/Users/caleb/Projects/BU Spring 2026/Module-B-semester-2/Milestone 3 EDA/ibm_hi_small_trans_cleaned.csv')
df

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Amount Paid,Is Laundering,Amount_Received_USD,Amount_Paid_USD,...,Payment Currency_Yuan,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Account_Same,Bank_Same
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.340000,3697.340000,0,3697.340000,3697.340000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.010000,0.010000,0,0.010000,0.010000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.570000,14675.570000,0,14675.570000,14675.570000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.970000,2806.970000,0,2806.970000,2806.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.970000,36682.970000,0,36682.970000,36682.970000,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,2022/09/10 23:57,54219,8148A6631,256398,8148A8711,0.154978,0.154978,0,3107.386389,3107.386389,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078341,2022/09/10 23:35,15,8148A8671,256398,8148A8711,0.108128,0.108128,0,2168.020464,2168.020464,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078342,2022/09/10 23:52,154365,8148A6771,256398,8148A8711,0.004988,0.004988,0,100.011894,100.011894,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
5078343,2022/09/10 23:46,256398,8148A6311,256398,8148A8711,0.038417,0.038417,0,770.280058,770.280058,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0,1


In [5]:
df.drop(columns=['Timestamp', 'From Bank', 'To Bank', 'Account', 'Account.1'], inplace=True)

## Undersampling

In [6]:
from collections import Counter


X = df.drop(columns='Is Laundering')
y = df['Is Laundering']
print("Original dataset shape:", Counter(y))
rus = RandomUnderSampler(random_state=42)
X_resampled, y_resampled = rus.fit_resample(X, y)

print("Resampled dataset shape:", Counter(y_resampled))

Original dataset shape: Counter({0: 5073168, 1: 5177})
Resampled dataset shape: Counter({0: 5177, 1: 5177})


## Split Data

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, 
                                                    y_resampled, 
                                                    test_size=0.2, 
                                                    stratify=y_resampled,  # Maintains class distribution
                                                    random_state=random_state
                                                    )

## Baseline Models
#### (No parameter optimization, feature scaling, or cross validation)

In [8]:
from sklearn.svm import SVC

# Using classifier since this is a fraud dataset with binary Y/N target
model = SVC()

In [9]:
model.fit(X_train, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [10]:
y_pred = model.predict(X_test)

In [11]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline SVM Classifier')

              precision    recall  f1-score   support

           0       0.50      1.00      0.67      1036
           1       0.50      0.00      0.00      1035

    accuracy                           0.50      2071
   macro avg       0.50      0.50      0.33      2071
weighted avg       0.50      0.50      0.33      2071



We see poor performance on the positive class, with 0% recall/f1. This could be due to the linear kernal choice or a lack of feature scaling. We can test this assumption during hyper-parameter tuning

## Feature Scaling

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [13]:
model.fit(X_train_scaled, y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [14]:
y_pred = model.predict(X_test_scaled)

In [15]:
print(classification_report(y_true=y_test, y_pred=y_pred))
report = classification_report(y_true=y_test, y_pred=y_pred, output_dict=True)
df_temp = pd.DataFrame(report).transpose()

classification_reports.append(df_temp)
classification_report_keys.append('Baseline SVM Classifier')

              precision    recall  f1-score   support

           0       0.88      0.89      0.89      1036
           1       0.89      0.88      0.89      1035

    accuracy                           0.89      2071
   macro avg       0.89      0.89      0.89      2071
weighted avg       0.89      0.89      0.89      2071



Once feature scaling is applied, recall on positive class jumps from 0% to 88%. This shows that feature scaling was critical for the model. We can attempt to further boost performance with hyperparameter tuning

## Parameter Tuning

In [16]:
from sklearn.model_selection import StratifiedKFold

# cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
cv = StratifiedKFold(n_splits=5) # Using stratified to maintain class distribution

In [23]:
def objective(trial):
    c = trial.suggest_float("C", 0.001, 1.0)
    kernel = trial.suggest_categorical("kernel", ["rbf", "sigmoid"])
    # degree = trial.suggest_int("degree", 2, 5) # degree of kernal function in case of 'poly'
    coef0 = trial.suggest_float("coef0", 0.0, 1.0) # Independent term in kernel function. It is only significant in ‘poly’ and ‘sigmoid’.
    # shrinking = trial.suggest_categorical("shrinking", [True, False])
    class_weight = trial.suggest_categorical("Class_weight", ['balanced', None])

    model = SVC(
        C=c,
        kernel=kernel,
        # degree=degree,
        coef0=coef0,
        # shrinking=shrinking,
        class_weight=class_weight,
        random_state=random_state,
        cache_size=1000 # FIX 3: Increase RAM cache from default 200MB to 1GB
    )
    
    scores = cross_val_score(
        model, 
        X=X_train_scaled, 
        y=y_train, 
        cv=cv,
        scoring='recall',
        n_jobs=-1,
    )
    recall = scores.mean()
    
    return recall


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

svc_scaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False)
top_5_svc_scaled_recall_optuna_results = study.trials_dataframe().sort_values("value", ascending=False).head(5)
display(top_5_svc_scaled_recall_optuna_results)

[I 2026-06-23 17:37:49,722] A new study created in memory with name: no-name-94ca5083-9951-4e3b-b312-c9d613741a0c
[I 2026-06-23 17:37:52,326] Trial 0 finished with value: 0.8401659644644907 and parameters: {'C': 0.676858026763721, 'kernel': 'sigmoid', 'coef0': 0.7460380971840528, 'Class_weight': 'balanced'}. Best is trial 0 with value: 0.8401659644644907.
[I 2026-06-23 17:37:54,807] Trial 1 finished with value: 0.8493490789788056 and parameters: {'C': 0.10380235686208915, 'kernel': 'rbf', 'coef0': 0.3377403186004688, 'Class_weight': None}. Best is trial 1 with value: 0.8493490789788056.
[I 2026-06-23 17:37:56,881] Trial 2 finished with value: 0.8599733104899098 and parameters: {'C': 0.9450886371817263, 'kernel': 'rbf', 'coef0': 0.8039369407045958, 'Class_weight': 'balanced'}. Best is trial 2 with value: 0.8599733104899098.
[I 2026-06-23 17:37:59,026] Trial 3 finished with value: 0.7998610164157969 and parameters: {'C': 0.6721101170345425, 'kernel': 'sigmoid', 'coef0': 0.022247547208508

,number,value,datetime_start,datetime_complete,duration,params_C,params_Class_weight,params_coef0,params_kernel,state
20,20,0.898619,2026-06-23 17:38:27.273556,2026-06-23 17:38:30.603196,0 days 00:00:03.329640,0.002131,None,0.213593,sigmoid,COMPLETE
43,43,0.880497,2026-06-23 17:39:05.270358,2026-06-23 17:39:07.034440,0 days 00:00:01.764082,0.190828,None,0.996798,sigmoid,COMPLETE
75,75,0.874706,2026-06-23 17:39:59.519510,2026-06-23 17:40:01.096558,0 days 00:00:01.577048,0.139761,balanced,0.779549,sigmoid,COMPLETE
30,30,0.873009,2026-06-23 17:38:44.306279,2026-06-23 17:38:46.059585,0 days 00:00:01.753306,0.196504,None,0.913644,sigmoid,COMPLETE
96,96,0.872526,2026-06-23 17:40:37.711138,2026-06-23 17:40:39.371542,0 days 00:00:01.660404,0.101529,None,0.809855,sigmoid,COMPLETE
